# SHAP Analysis & Lift Curve

Two business-facing analyses on the tuned XGBoost model:
- **SHAP**: what drives each prediction — globally and at the individual customer level
- **Lift curve**: how much better than random is the model at targeting converters

## 1. Setup & Retrain Tuned Model

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shap

from src.data import load_config, load_raw, clean, split
from src.features import build_feature_matrix
from src.model import tune_hyperparameters, train_model, evaluate

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)
shap.initjs()

cfg       = load_config("../config.yaml")
TARGET    = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

# Load, clean, split
df = clean(load_raw("../data/raw"))
train_df, test_df = split(
    df,
    test_size=cfg["data"]["test_size"],
    random_state=cfg["split"]["random_state"],
)

# Engineered feature matrix
train_eng, test_eng = build_feature_matrix(train_df, test_df, target_col=TARGET)

X_train = train_eng.drop(columns=DROP_COLS)
X_test  = test_eng.drop(columns=DROP_COLS)
y_train = train_eng[TARGET]
y_test  = test_eng[TARGET]

print(f"Features ({X_train.shape[1]}): {X_train.columns.tolist()}")

In [ ]:
best_params = tune_hyperparameters(
    X_train, y_train,
    n_trials=cfg["tuning"]["n_trials"],
    cv_folds=cfg["tuning"]["cv_folds"],
    random_state=cfg["split"]["random_state"],
)
model  = train_model(X_train, y_train, best_params)
probs  = model.predict_proba(X_test)[:, 1]
result = evaluate("XGBoost (tuned)", y_test, probs)
print(f"\nTest PR-AUC: {result['pr_auc']}  ROC-AUC: {result['roc_auc']}")

## 2. SHAP Values

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print(f"SHAP values shape: {shap_values.shape}")

## 3. Global Feature Importance (mean |SHAP|)

In [ ]:
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test.columns
).sort_values()

plt.figure(figsize=(9, 6))
colors = ["#DD8452" if v == mean_shap.max() else "#4C72B0" for v in mean_shap.values]
plt.barh(mean_shap.index, mean_shap.values, color=colors)
plt.title("Global Feature Importance — mean |SHAP value|")
plt.xlabel("Mean |SHAP|")
plt.tight_layout()

## 4. SHAP Beeswarm Plot

In [ ]:
# Sample 2000 rows for speed
sample_idx  = np.random.RandomState(42).choice(len(X_test), size=2000, replace=False)
X_sample    = X_test.iloc[sample_idx]
sv_sample   = shap_values[sample_idx]

shap.summary_plot(sv_sample, X_sample, plot_type="dot", show=True)

## 5. Individual Prediction Explanation

In [ ]:
# Pick one high-probability converter and one non-converter
converter_idx     = np.where((y_test.values == 1) & (probs > 0.7))[0][0]
non_converter_idx = np.where((y_test.values == 0) & (probs < 0.1))[0][0]

for label, idx in [("High-probability converter", converter_idx),
                   ("Predicted non-converter",    non_converter_idx)]:
    print(f"\n── {label} (prob={probs[idx]:.3f}) ──")
    shap.waterfall_plot(
        shap.Explanation(
            values        = shap_values[idx],
            base_values   = explainer.expected_value,
            data          = X_test.iloc[idx].values,
            feature_names = X_test.columns.tolist(),
        ),
        show=True,
    )

## 6. SHAP Dependence — Top 2 Features

In [ ]:
top2 = mean_shap.tail(2).index.tolist()[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, top2):
    shap.dependence_plot(feat, shap_values, X_test, ax=ax, show=False)
    ax.set_title(f"SHAP dependence: {feat}")
plt.tight_layout()

## 7. Lift Curve & Decile Analysis

In [ ]:
lift_df = (
    pd.DataFrame({"prob": probs, "actual": y_test.values})
    .sort_values("prob", ascending=False)
    .reset_index(drop=True)
)

lift_df["decile"] = pd.qcut(lift_df.index, q=10, labels=range(1, 11))

decile_stats = lift_df.groupby("decile", observed=True).agg(
    n_customers  = ("actual", "count"),
    n_converters = ("actual", "sum"),
    conv_rate    = ("actual", "mean"),
).reset_index()

overall_rate = y_test.mean()
decile_stats["lift"] = decile_stats["conv_rate"] / overall_rate
decile_stats["cum_converters_pct"] = (
    decile_stats["n_converters"].cumsum() / y_test.sum() * 100
)
decile_stats["cum_customers_pct"] = (
    decile_stats["n_customers"].cumsum() / len(y_test) * 100
)

print(decile_stats[["decile","conv_rate","lift","cum_converters_pct"]].to_string(index=False))

## 8. Lift & Cumulative Gain Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lift chart
axes[0].bar(decile_stats["decile"].astype(str), decile_stats["lift"], color="#4C72B0")
axes[0].axhline(1.0, color="grey", linestyle="--", label="Random (lift=1)")
axes[0].set_title("Lift by Decile\n(top decile = highest scored customers)")
axes[0].set_xlabel("Decile (1 = top scored)")
axes[0].set_ylabel("Lift")
axes[0].legend()
for i, v in enumerate(decile_stats["lift"]):
    axes[0].text(i, v + 0.05, f"{v:.1f}x", ha="center", fontsize=8)

# Cumulative gain chart
axes[1].plot(
    decile_stats["cum_customers_pct"],
    decile_stats["cum_converters_pct"],
    marker="o", color="#4C72B0", label="Model"
)
axes[1].plot([0, 100], [0, 100], "grey", linestyle="--", label="Random")
axes[1].set_title("Cumulative Gain Chart")
axes[1].set_xlabel("% Customers contacted (ranked by score)")
axes[1].set_ylabel("% Converters captured")
axes[1].legend()

# Annotate 20% and 30% marks
for pct in [20, 30]:
    row = decile_stats[decile_stats["cum_customers_pct"] <= pct + 5].iloc[-1]
    axes[1].annotate(
        f"{row['cum_converters_pct']:.0f}% converters\n@ top {pct}%",
        xy=(row["cum_customers_pct"], row["cum_converters_pct"]),
        xytext=(row["cum_customers_pct"] + 5, row["cum_converters_pct"] - 10),
        arrowprops=dict(arrowstyle="->", color="black"),
        fontsize=8,
    )

plt.tight_layout()

## 9. Business Impact Summary

In [ ]:
top1 = decile_stats.iloc[0]
top2 = decile_stats.iloc[1]
top3 = decile_stats.iloc[2]

print("=" * 60)
print("BUSINESS IMPACT SUMMARY")
print("=" * 60)
print(f"Overall conversion rate (baseline):  {overall_rate:.1%}")
print()
print(f"Top 10% of scored customers:")
print(f"  Conversion rate : {top1['conv_rate']:.1%}  ({top1['lift']:.1f}x lift)")
print(f"  Converters captured : {top1['cum_converters_pct']:.0f}% of all converters")
print()
print(f"Top 20% of scored customers:")
print(f"  Conversion rate : {top2['conv_rate']:.1%}  ({top2['lift']:.1f}x lift)")
cum20 = decile_stats.iloc[:2]["n_converters"].sum() / y_test.sum() * 100
print(f"  Converters captured : {cum20:.0f}% of all converters")
print()
print(f"Top 30% of scored customers:")
print(f"  Conversion rate : {top3['conv_rate']:.1%}  ({top3['lift']:.1f}x lift)")
cum30 = decile_stats.iloc[:3]["n_converters"].sum() / y_test.sum() * 100
print(f"  Converters captured : {cum30:.0f}% of all converters")
print()
print("Takeaway: By targeting the top 20% of customers by score, the")
print(f"model captures ~{cum20:.0f}% of all converters while contacting")
print(f"only 20% of the customer base — {cum20/20:.1f}x more efficient than random outreach.")